<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "288"
iso_country = "Ghana"
year = 2025
import_portfolio = "h6_ag6_by_sea_FM.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 114.210*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Ghana': 

Number of trade records present in dataset: 45551
Number of partner countries present in dataset: 186
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$18255.9m
Total weight within the dataset: 23070.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,288,Ghana,699,India,Foreign Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,4707.447,NaN,4707.447,kg,8000.0,8000.0,7785.646593,14742.0,9160.251282,71318.479187,['Container'],['Container'],235.253715,234.266303,234.333817,234.333817,234.992407,234.992407,247.314109,471.494818,275.138323,250.865111,251.890825,259.882545,241.583756,445.500522,284.726059,261.508003,255.725556,268.726900,236.563933,492.629995,357.970724,310.82919,287.470165,280.934441,231.607400,523.509574,416.801721,391.037834,304.317363,302.325314,226.741281,533.033725,497.737642,505.967596,331.102623,318.075338,221.783062,524.076659,506.225676,533.977696,395.157285,324.102127,217.110723,466.205062,475.936838,498.848715,375.806898,328.103424,212.37363,416.943590,428.515401,498.771357,408.615861,330.718702,2322
1,1,288,Ghana,156,China,Foreign Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,793.073,NaN,793.073,kg,85.0,2000.0,1311.663435,20237.0,12574.684927,16493.754422,['Container'],['Container'],54.406895,54.178537,54.194151,54.194151,54.346462,54.346462,57.196090,109.042142,63.630969,58.017327,58.254543,60.102780,55.870837,103.030467,65.848315,60.478698,59.141397,62.148205,54.709908,113.930054,82.787537,71.88516,66.482942,64.971431,53.563615,121.071544,96.393323,90.434935,70.379177,69.918478,52.438233,123.274185,115.111294,117.014627,76.573777,73.560970,51.291550,121.202693,117.074313,123.492496,91.387636,74.954780,50.210983,107.818785,110.069444,115.368251,86.912491,75.880156,49.11544,96.426133,99.102335,115.350360,94.500188,76.484988,2322


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $2581.6m (14.1%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               365022685.8  763597804.8      2.0      3.3
11 | Products of the mill  323170876.2  606298026.3      1.8      2.6
02 | Meat and edible meat  266937854.9  141485773.5      1.5      0.6
15 | Animal, vegetable or  233479226.4  234057764.1      1.3      1.0
05 | Animal originated pr  216953170.1   51048944.0      1.2      0.2
03 | Fish and crustaceans  201113320.7   89939879.7      1.1      0.4
22 | Beverages, spirits a  159242665.0  130711280.9      0.9      0.6
17 | Sugars and sugar con  153388237.6  300932404.0      0.8      1.3
21 | Miscellaneous edible  139155318.0   34655386.3      0.8      0.2
23 | Food industries, res  128438891.9  215863085.6      0.7      0.9
19 | Preparations of cere  110358764.4   44268990.7      0

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $2375.8m (13.0%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Agriculture       1.553353e+09  1.828900e+09      8.5      7.9
Food & Beverages  7.900928e+08  7.598259e+08      4.3      3.3
Fishing           3.237491e+07  9.073043e+06      0.2      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $2226.1m (12.2%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     753160250.9  802304498.8      4.1      3.5
PCR - Processed rice           307487117.6  534105123.2      1.7      2.3
VOL - Vegetable oils and fats  299238605.3  389681407.7      1.6      1.7
OMT - Meat products n.e.c.     253597192.3  134576496.9      1.4      0.6
OAP - Animal products n.e.c.   219017695.7   52599667.0      1.2      0.2
SGR - Sugar                    127145764.5  280236634.1      0.7      1.2
MIL - Dairy products            97980556.7   42919931.7      0.5      0.2
V_F - Vegetables, fruit, nuts   42842162.4   41452035.5      0.2      0.2
GRO - Cereal grains n.e.c.      41115858.0  170072767.1      0.2      0.7
CMT - Bovine meat prods         32631146.5   13650037.8      0.2      0.1
FSH - Fishing                   22192088.5   

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 14.1% 
GTAP 12.2% 
EORA 13.0%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46375/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,288,Ghana,699,India,Foreign Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,4707.447,NaN,4707.447,kg,8000.0,8000.0,7785.646593,14742.0,9160.251282,71318.479187,['Container'],['Container'],235.253715,234.266303,234.333817,234.333817,234.992407,234.992407,247.314109,471.494818,275.138323,250.865111,251.890825,259.882545,241.583756,445.500522,284.726059,261.508003,255.725556,268.726900,236.563933,492.629995,357.970724,310.82919,287.470165,280.934441,231.607400,523.509574,416.801721,391.037834,304.317363,302.325314,226.741281,533.033725,497.737642,505.967596,331.102623,318.075338,221.783062,524.076659,506.225676,533.977696,395.157285,324.102127,217.110723,466.205062,475.936838,498.848715,375.806898,328.103424,212.37363,416.943590,428.515401,498.771357,408.615861,330.718702,2322,Processed Cereals
1,1,288,Ghana,156,China,Foreign Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,793.073,NaN,793.073,kg,85.0,2000.0,1311.663435,20237.0,12574.684927,16493.754422,['Container'],['Container'],54.406895,54.178537,54.194151,54.194151,54.346462,54.346462,57.196090,109.042142,63.630969,58.017327,58.254543,60.102780,55.870837,103.030467,65.848315,60.478698,59.141397,62.148205,54.709908,113.930054,82.787537,71.88516,66.482942,64.971431,53.563615,121.071544,96.393323,90.434935,70.379177,69.918478,52.438233,123.274185,115.111294,117.014627,76.573777,73.560970,51.291550,121.202693,117.074313,123.492496,91.387636,74.954780,50.210983,107.818785,110.069444,115.368251,86.912491,75.880156,49.11544,96.426133,99.102335,115.350360,94.500188,76.484988,2322,Processed Cereals


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $1564.9m of $18255.9m (8.6%)
Volume: 2045.5 of 23070.3 million kilograms (8.9%)
Value-to-Weight: 0.77 from 0.79 US$ per kg (+-3.3%)
Average distance to market (volume-weighted): 6614.4 from 6631.2 nm (+-0.3%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals   1164  856786942.4  1.212400e+09      4.7      5.3
1        Meat and Prep    229  236740801.7  1.309652e+08      1.3      0.6
2      Sugar and Honey    215  153407352.6  3.009379e+08      0.8      1.3
3                Dairy    319   97299359.0  4.195187e+07      0.5      0.2
4        Fats and Oils    296   91409402.5  6.682714e+07      0.5      0.3
5  Unprocessed Cereals     66   57467238.5  2.293740e+08      0.3      1.0
6           Vegetables    712   37068758.1  3.622430e+07      0.2      0.2
7               Fruits    389   34673253.5  2.683024e+07      0.2      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,9.247397e+07,1.746016e+08,1.026012e+08,9.371764e+07,9.409261e+07,9.709442e+07,8.491942e+07,1.973617e+08,1.845502e+08,1.877861e+08,1.233965e+08,1.184808e+08,7.964247e+07,1.549098e+08,1.592109e+08,1.852224e+08,1.518590e+08,1.231280e+08
1,Fats and Oils,1.594411e+08,2.971738e+08,1.762730e+08,1.613560e+08,1.619861e+08,1.672226e+08,1.467274e+08,3.357203e+08,3.146320e+08,3.205972e+08,2.118839e+08,2.032338e+08,1.378482e+08,2.646942e+08,2.719964e+08,3.163389e+08,2.596985e+08,2.110349e+08
2,Fruits,3.621355e+07,6.854032e+07,4.019803e+07,3.672770e+07,3.687372e+07,3.803214e+07,3.323975e+07,7.757675e+07,7.243857e+07,7.367653e+07,4.836410e+07,4.645214e+07,3.116336e+07,6.090454e+07,6.255353e+07,7.271745e+07,5.951147e+07,4.829120e+07
3,Meat and Prep,2.182972e+08,4.161751e+08,2.428568e+08,2.214315e+08,2.223369e+08,2.293910e+08,2.001381e+08,4.704938e+08,4.393389e+08,4.466033e+08,2.922549e+08,2.807561e+08,1.874562e+08,3.680243e+08,3.782384e+08,4.402513e+08,3.606736e+08,2.919160e+08
4,Processed Cereals,3.088927e+09,5.685219e+09,3.399488e+09,3.121837e+09,3.134045e+09,3.235620e+09,2.848697e+09,6.487373e+09,6.080444e+09,6.201331e+09,4.123318e+09,3.913685e+09,2.680948e+09,5.121826e+09,5.255696e+09,6.086524e+09,5.015233e+09,4.059102e+09
5,Sugar and Honey,4.252144e+08,7.647112e+08,4.647173e+08,4.287331e+08,4.303636e+08,4.445057e+08,3.936129e+08,8.782315e+08,8.251540e+08,8.434132e+08,5.665919e+08,5.335782e+08,3.715458e+08,6.974899e+08,7.149948e+08,8.252805e+08,6.826734e+08,5.524774e+08
6,Unprocessed Cereals,3.683298e+08,6.527881e+08,4.008009e+08,3.707940e+08,3.721813e+08,3.845474e+08,3.417525e+08,7.529178e+08,7.087366e+08,7.252928e+08,4.906099e+08,4.593372e+08,3.231940e+08,6.001848e+08,6.148571e+08,7.081095e+08,5.875039e+08,4.750448e+08
7,Vegetables,9.054851e+07,1.672339e+08,9.975832e+07,9.163348e+07,9.198262e+07,9.488813e+07,8.344968e+07,1.900479e+08,1.777400e+08,1.813943e+08,1.201251e+08,1.149955e+08,7.849609e+07,1.503298e+08,1.542422e+08,1.788456e+08,1.465928e+08,1.194556e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.84,0.10,0.01,0.02,0.05,1.16,1.02,1.06,0.40,0.34,0.77,0.82,1.09,0.74,0.45
1,Fats and Oils,1.51,0.18,0.02,0.03,0.09,2.07,1.84,1.90,0.71,0.62,1.39,1.47,1.95,1.33,0.80
2,Fruits,0.93,0.11,0.01,0.02,0.05,1.28,1.13,1.17,0.44,0.38,0.86,0.91,1.20,0.82,0.49
3,Meat and Prep,0.84,0.10,0.01,0.02,0.05,1.14,1.01,1.04,0.39,0.34,0.76,0.81,1.07,0.73,0.44
4,Processed Cereals,3.03,0.36,0.04,0.05,0.17,4.25,3.77,3.91,1.49,1.24,2.85,3.01,3.97,2.72,1.61
5,Sugar and Honey,2.21,0.26,0.02,0.03,0.13,3.16,2.81,2.93,1.13,0.91,2.12,2.24,2.96,2.03,1.18
6,Unprocessed Cereals,4.95,0.57,0.04,0.07,0.28,7.15,6.39,6.67,2.59,2.05,4.82,5.08,6.70,4.60,2.64
7,Vegetables,2.07,0.25,0.03,0.04,0.12,2.88,2.54,2.64,0.99,0.85,1.94,2.04,2.71,1.84,1.10
